# StrOutputParser - LLM 출력을 문자열로 변환하기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- LLM이 반환하는 `AIMessage` 객체와 순수 문자열의 차이를 설명할 수 있어요
- `StrOutputParser`를 LCEL(LangChain Expression Language) 파이프라인에 연결할 수 있어요
- `stream()` 메서드로 실시간 스트리밍 출력을 구현할 수 있어요

## 사전 지식

- **LCEL 파이프라인**: `prompt | model | parser` 형태로 컴포넌트를 연결하는 LangChain의 표현식 언어예요
- **AIMessage**: LangChain이 LLM 응답을 담는 객체로, 텍스트 외에 메타데이터(토큰 수, 모델명 등)도 함께 저장해요

## OutputParser 시리즈 전체 흐름

```mermaid
flowchart LR
    LLM["LLM 응답<br/>(AIMessage)"] --> P1["StrOutputParser<br/>▶ 노트북 01"]
    LLM --> P2["PydanticOutputParser<br/>▶ 노트북 02"]
    LLM --> P3["JsonOutputParser<br/>▶ 노트북 02"]
    LLM --> P4["CommaSeparatedList<br/>▶ 노트북 03"]
    LLM --> P5["datetime / Enum<br/>▶ 노트북 04"]
    LLM --> P6["PandasDataFrame<br/>▶ 노트북 05"]
    P1 --> O1["str"]
    P2 --> O2["Pydantic 객체"]
    P3 --> O3["dict"]
    P4 --> O4["list"]
    P5 --> O5["datetime / Enum"]
    P6 --> O6["DataFrame 조회"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class LLM input
    class P1,P2,P3,P4,P5,P6 process
    class O1,O2,O3,O4,O5,O6 output
```

## StrOutputParser의 핵심 역할

```mermaid
flowchart LR
    A["PromptTemplate"] -->|"입력 변수 채움"| B["ChatOpenAI<br/>(LLM)"]
    B -->|"AIMessage 반환"| C["StrOutputParser"]
    C -->|"str 반환"| D["최종 결과"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C process
    class D output
```

`StrOutputParser`는 LangChain에서 가장 기본적이고 자주 사용되는 출력 파서예요. `AIMessage` 객체에서 텍스트 내용만 꺼내 깔끔한 문자열로 돌려줘요.

## 언제 사용하면 좋을까요?

- LLM 응답을 **단순 텍스트**로만 쓸 때
- 복잡한 구조 없이 **빠르게 결과를 확인**하고 싶을 때
- **대화형 응답**을 바로 화면에 출력할 때

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 기본 사용법

가장 간단한 형태의 `StrOutputParser` 사용 예제입니다.

> 🎯 **강의 포인트**: `StrOutputParser`는 LangChain에서 가장 많이 쓰이는 파서예요. 복잡한 설정 없이 `StrOutputParser()` 한 줄로 AIMessage를 순수 문자열로 변환해 줍니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1단계: 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: 프롬프트 템플릿 정의
prompt = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해주세요.")

# 3단계: StrOutputParser 생성
output_parser = StrOutputParser()

# 4단계: 체인 구성 (프롬프트 | 모델 | 파서)
# 🎯 강의 포인트: prompt | model | output_parser 가 LCEL 파이프라인의 기본 형태
chain = prompt | model | output_parser

# 5단계: 실행
result = chain.invoke({"topic": "양자 컴퓨팅"})
print(result)

## StrOutputParser 없이 사용하는 경우

파서를 사용하지 않으면 `AIMessage` 객체가 반환됩니다.


---

## 파서 없이 vs. 파서 사용: 직접 비교해요

`StrOutputParser`를 쓰지 않으면 LLM은 `AIMessage` 객체를 그대로 돌려줘요. 두 방식을 나란히 비교해볼게요.

> **실무 팁**: 다운스트림(하위) 코드에서 `.content`를 일일이 호출하는 대신 파서를 체인에 포함시키는 편이 훨씬 깔끔해요.

In [ ]:
# ---------------------------------------------------
# 파서 유무에 따른 출력 타입 직접 비교
# ---------------------------------------------------

# 파서 없이 체인 구성
# ⚠️ 주의: AIMessage 객체를 그냥 쓰면 다운스트림 코드마다 .content를 호출해야 함
chain_without_parser = prompt | model

# 실행
result_without_parser = chain_without_parser.invoke({"topic": "양자 컴퓨팅"})

print("=" * 60)
print("📦 파서 없이 실행한 결과")
print("=" * 60)
print(f"타입: {type(result_without_parser)}")
print(f"\n내용:\n{result_without_parser}")
print()

print("=" * 60)
print("📝 텍스트만 추출하려면")
print("=" * 60)
# .content: AIMessage 객체에서 텍스트만 꺼내는 속성
print(result_without_parser.content)

## StrOutputParser를 사용하는 경우

파서를 사용하면 깔끔한 문자열만 반환됩니다.


In [ ]:
# ---------------------------------------------------
# StrOutputParser를 추가한 체인 - 결과 타입 비교
# ---------------------------------------------------

# 파서를 포함한 체인
# StrOutputParser(): AIMessage에서 .content 텍스트만 추출
chain_with_parser = prompt | model | output_parser

# 실행
result_with_parser = chain_with_parser.invoke({"topic": "양자 컴퓨팅"})

print("=" * 60)
print("✅ 파서를 사용한 결과")
print("=" * 60)
# 타입이 str로 바뀐 것을 확인
print(f"타입: {type(result_with_parser)}")
print(f"\n내용:\n{result_with_parser}")

---

## 스트리밍: 응답이 생성되는 즉시 출력해요

`stream()` 메서드를 사용하면 LLM이 토큰을 생성하는 대로 즉시 화면에 출력할 수 있어요. 사용자 경험이 크게 향상되는 기법이에요.

> ⚠️ **자주 하는 실수**: `invoke()`는 전체 응답을 기다린 뒤 한 번에 반환해요. `stream()`은 생성 즉시 청크(chunk) 단위로 반환하므로, 긴 응답일수록 체감 속도 차이가 커요.

> 💡 **실무 팁**: 챗봇이나 실시간 응답 UI를 만들 때는 반드시 `stream()`을 사용하세요. `print(chunk, end="", flush=True)` 패턴을 기억해두면 어디서나 바로 쓸 수 있어요.

## 스트리밍 출력

`StrOutputParser`는 스트리밍을 지원하여 실시간으로 응답을 출력할 수 있습니다.


---

## 실무 예제: 다양한 시나리오에서 활용해요

`StrOutputParser`는 단순하지만 번역, 감정 분석, 질의응답 등 거의 모든 텍스트 생성 태스크에 바로 사용할 수 있어요.

> 🔑 **핵심 개념**: `PromptTemplate.from_template()`은 문자열에서 바로 템플릿을 만드는 편의 메서드예요. 중괄호 `{변수명}` 로 입력 변수를 선언합니다.

In [ ]:
# 스트리밍을 위한 프롬프트
streaming_prompt = PromptTemplate.from_template(
    "{topic}에 대해 3가지 핵심 특징을 설명해주세요."
)

# 체인 구성
streaming_chain = streaming_prompt | model | output_parser

print("=" * 60)
print("🔄 스트리밍 출력 (실시간)")
print("=" * 60)

# 스트리밍 실행 - 각 청크를 실시간으로 출력
# 💡 팁: end=""는 줄바꿈 없이 이어 붙이고, flush=True는 버퍼를 즉시 화면에 내보냄
for chunk in streaming_chain.stream({"topic": "머신러닝"}):
    print(chunk, end="", flush=True)

print("\n" + "=" * 60)

## 핵심 요약

| 항목 | 설명 |
|------|------|
| **역할** | `AIMessage` 객체에서 텍스트만 추출해 `str`로 반환 |
| **설정** | 별도 스키마 불필요, `StrOutputParser()` 한 줄로 끝 |
| **스트리밍** | `stream()` 메서드와 함께 사용해 실시간 출력 가능 |
| **체인 위치** | 항상 파이프라인 맨 끝 `prompt \| model \| parser` |

### StrOutputParser를 선택하는 기준

```mermaid
flowchart TD
    Q1{"출력 형태가<br/>단순 텍스트인가요?"}
    Q2{"구조화된 데이터<br/>(객체·딕셔너리·리스트)가<br/>필요한가요?"}
    A1["StrOutputParser 사용"]
    A2["노트북 02~05의<br/>구조화 파서로 이동"]

    Q1 -->|예| A1
    Q1 -->|아니오| Q2
    Q2 -->|예| A2

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class Q1,Q2 process
    class A1 output
    class A2 input
```

## 다음 노트북 예고

**노트북 02 - 구조화된 출력 파싱** 에서는 단순 텍스트를 넘어 Pydantic 모델과 JSON 딕셔너리로 변환하는 방법을 배워요. 타입 검증과 필드 제약조건까지 LangChain이 자동으로 처리해줘요.

## 다양한 입력 형식 처리

`StrOutputParser`는 다양한 형태의 LLM 응답을 문자열로 변환할 수 있습니다.


In [ ]:
# ---------------------------------------------------
# 다양한 시나리오에서 StrOutputParser 활용
# ---------------------------------------------------

# 1. 간단한 질문
# PromptTemplate.from_template(): 문자열에서 바로 템플릿 생성 (편의 메서드)
simple_chain = (
    PromptTemplate.from_template("질문: {question}") | model | StrOutputParser()
)

answer1 = simple_chain.invoke({"question": "Python의 주요 특징은?"})
print("💡 질문 1 답변:")
print(answer1)
print()

# 2. 복잡한 분석 요청
analysis_chain = (
    PromptTemplate.from_template(
        "다음 텍스트의 감정을 분석하세요:\n\n{text}\n\n감정 분석 결과:"
    )
    | model
    | StrOutputParser()
)

answer2 = analysis_chain.invoke(
    {"text": "오늘 회의는 정말 생산적이었어요. 팀원들의 아이디어가 훌륭했습니다!"}
)
print("🎭 감정 분석 결과:")
print(answer2)
print()

# 3. 번역 요청
translation_chain = (
    PromptTemplate.from_template("다음 문장을 영어로 번역하세요: {korean_text}")
    | model
    | StrOutputParser()
)

answer3 = translation_chain.invoke({"korean_text": "인공지능은 미래를 바꿀 것입니다."})
print("🌐 번역 결과:")
print(answer3)

## 💡 핵심 정리

```python
# StrOutputParser의 핵심 역할

# 파서 없이:
response = model.invoke("안녕하세요")
# 결과: AIMessage(content="안녕하세요! 무엇을 도와드릴까요?", ...)

# StrOutputParser 사용:
response = (model | StrOutputParser()).invoke("안녕하세요")
# 결과: "안녕하세요! 무엇을 도와드릴까요?"
```

### 주요 장점

- ✅ **간편함**: 별도 설정 없이 바로 사용 가능
- ✅ **깔끔한 출력**: 순수 문자열만 반환
- ✅ **스트리밍 지원**: 실시간 응답 가능
- ✅ **체인 통합**: LCEL 파이프라인에 자연스럽게 통합

### 다음 단계

단순 문자열이 아닌 **구조화된 데이터**가 필요하다면:
- `PydanticOutputParser` - 객체 형태로 변환
- `JsonOutputParser` - JSON 형태로 변환
- `CommaSeparatedListOutputParser` - 리스트 형태로 변환


### 연습 1: 다국어 번역 체인 만들기

`StrOutputParser`를 사용하여 한국어 문장을 **영어, 일본어, 중국어** 3가지 언어로 번역하는 체인을 만들어보세요.

**요구사항:**
- `ChatPromptTemplate`을 사용하여 시스템 메시지와 사용자 메시지를 포함하는 프롬프트를 구성하세요
- 입력 변수: `text` (번역할 한국어 문장), `language` (번역 대상 언어)
- 테스트 문장: "인공지능이 세상을 바꾸고 있습니다."
- 3가지 언어로 각각 번역 결과를 출력하세요

In [ ]:
# ============================================================
# TODO: 다국어 번역 체인 만들기
# 힌트: ChatPromptTemplate.from_messages()로 시스템/사용자 메시지를 구성하고,
#       StrOutputParser()를 체인 끝에 연결하세요
# 예상 결과: 동일한 한국어 문장이 영어/일본어/중국어로 번역되어 출력
# ============================================================

# 연습 1 풀이

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1단계: 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: 번역용 프롬프트 구성
# ChatPromptTemplate.from_messages(): 시스템/사용자 역할을 명시적으로 분리
translation_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 전문 번역가입니다. 주어진 한국어 문장을 정확하게 번역하세요."),
    ("user", "다음 한국어 문장을 {language}로 번역해주세요:\n\n{text}")
])

# 3단계: 체인 구성: 프롬프트 | 모델 | StrOutputParser
translation_chain = translation_prompt | model | StrOutputParser()

# 4단계: 번역할 문장 준비
source_text = "인공지능이 세상을 바꾸고 있습니다."

# 5단계: 3가지 언어로 각각 번역 실행
languages = ["영어", "일본어", "중국어"]

print("=" * 60)
print(f"원문: {source_text}")
print("=" * 60)

for lang in languages:
    result = translation_chain.invoke({"text": source_text, "language": lang})
    print(f"\n[{lang}] {result}")

### 연습 2: 스트리밍으로 이야기 생성하기

`StrOutputParser`와 `stream()` 메서드를 사용하여 짧은 이야기를 **실시간으로 출력**하는 체인을 만들어보세요.

**요구사항:**
- `PromptTemplate`을 사용하여 주제(`topic`)와 문장 수(`sentences`)를 입력받는 프롬프트를 만드세요
- `stream()` 메서드로 실시간 출력하세요
- 테스트: 주제 "우주 탐험", 문장 수 5

In [ ]:
# ============================================================
# TODO: 스트리밍으로 이야기 생성하기
# 힌트: PromptTemplate에 topic과 sentences 변수를 정의하고,
#       chain.stream()으로 실행한 뒤 각 chunk를 출력하세요
# 예상 결과: 이야기가 토큰 단위로 실시간 출력됨
# ============================================================

# 연습 2 풀이

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1단계: 모델 초기화 (temperature=0.7: 창의적인 이야기 생성을 위해 다양성 높임)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 2단계: 이야기 생성 프롬프트
story_prompt = PromptTemplate.from_template(
    "{topic}에 대한 짧은 이야기를 {sentences}문장으로 작성해주세요. "
    "흥미롭고 생동감 있게 써주세요."
)

# 3단계: 체인 구성
story_chain = story_prompt | model | StrOutputParser()

# 4단계: stream()으로 실시간 출력
# stream(): 응답을 청크(chunk) 단위로 즉시 반환
# end="": 줄바꿈 없이 이어서 출력 / flush=True: 버퍼를 즉시 출력
print("=" * 60)
print("우주 탐험 이야기 (스트리밍)")
print("=" * 60)

for chunk in story_chain.stream({"topic": "우주 탐험", "sentences": 5}):
    print(chunk, end="", flush=True)

print("\n" + "=" * 60)

### 연습 3: StrOutputParser 유무 비교 실험

동일한 질문에 대해 `StrOutputParser`를 **사용한 경우**와 **사용하지 않은 경우**의 결과를 비교해보세요.

**요구사항:**
- 질문: "Python과 JavaScript의 주요 차이점을 한 문장으로 설명해주세요."
- 파서 없이 실행한 결과의 `type`, `content`, `response_metadata`를 출력하세요
- 파서를 사용한 결과의 `type`과 내용을 출력하세요
- 두 결과의 차이를 확인하세요

In [ ]:
# ============================================================
# TODO: StrOutputParser 유무 비교 실험
# 힌트: 파서 없는 체인은 prompt | model, 파서 있는 체인은 prompt | model | StrOutputParser()
# 예상 결과: 파서 없이는 AIMessage, 파서 사용 시 str 타입이 출력됨
# ============================================================

# 연습 3 풀이

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1단계: 모델 및 프롬프트 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = PromptTemplate.from_template(
    "질문: {question}"
)

question = "Python과 JavaScript의 주요 차이점을 한 문장으로 설명해주세요."

# 2단계: 파서 없이 실행
chain_without_parser = prompt | model
result_without = chain_without_parser.invoke({"question": question})

print("=" * 60)
print("[파서 없이 실행한 결과]")
print("=" * 60)
print(f"타입: {type(result_without)}")
# .content: 텍스트 본문 / .response_metadata: 토큰 수·모델명 등 메타데이터
print(f"content: {result_without.content}")
print(f"메타데이터 키: {list(result_without.response_metadata.keys())}")

print()

# 3단계: 파서를 사용한 실행
# StrOutputParser: .content만 꺼내서 str로 반환
chain_with_parser = prompt | model | StrOutputParser()
result_with = chain_with_parser.invoke({"question": question})

print("=" * 60)
print("[파서를 사용한 결과]")
print("=" * 60)
print(f"타입: {type(result_with)}")
print(f"내용: {result_with}")

print()
print("=" * 60)
print("[비교 결과]")
print("=" * 60)
print(f"파서 없이: {type(result_without).__name__} 객체 (content, metadata 등 포함)")
print(f"파서 사용: {type(result_with).__name__} (순수 텍스트만 반환)")
print(f"내용 동일: {result_without.content == result_with}")